# NE-EpiGuard — Feature Engineering + Model Training + SHAP
## Notebook 2

---

### Pipeline Overview
```
Feature Engineering
      ↓
Baseline Model Testing (6 models + MLflow)
      ↓
Hyperparameter Tuning (top 3 models)
      ↓
Stacking Ensemble
      ↓
SHAP Explainability
      ↓
Save Final Model
```
---

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

import dill
import os


#ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, recall_score
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import shap


# MLFlow 
import mlflow
import mlflow.sklearn


# Directories
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('mlflow', exist_ok=True)


# MLflow setup
mlflow.set_tracking_uri('file:///D:/NE-EpiGuard/mlflow')
mlflow.set_experiment('NE-EpiGuard-Disease-Prediction')

print('All imports successful')
print('MLflow tracking URI:', mlflow.get_tracking_uri())

All imports successful
MLflow tracking URI: file:///D:/NE-EpiGuard/mlflow


## 2. Load Data

In [28]:
data = pd.read_csv('Data/northeast_waterborne.csv')
print('Shape:', data.shape)
print('Target Distribution:')
print(data['disease'].value_counts())

Shape: (405074, 39)
Target Distribution:
disease
No_Disease       113645
Typhoid           55454
Dysentery         49956
Giardiasis        46087
Cholera           42565
Hepatitis_E       34769
Hepatitis_A       34063
Leptospirosis     28535
Name: count, dtype: int64


## 3. Feature Engineering

### 3.1 Drop Weak/Redundant Columns

In [29]:
columns_drop = [
    'district', 'latitude', 'longitude',
    'population_density', 'gender'  
]

data = data.drop(columns=columns_drop)

print('Dropped:', columns_drop)
print('Remaining shape:', data.shape)

Dropped: ['district', 'latitude', 'longitude', 'population_density', 'gender']
Remaining shape: (405074, 34)


### 3.2 Ordinal Encoding (risk-ordered categories)

In [30]:
# Handwash Column 
hand_map = {
    'Never': 0, 'Sometimes': 1,
    'Always': 2
}

data['handwashing_practice'] = data['handwashing_practice'].map(hand_map)


# Water Treatment
water_map = {
    'Untreated': 0, 'Boiled': 1,
    'Filtered': 2, 'Chlorinated': 3
}

data['water_treatment'] = data['water_treatment'].map(water_map)


# Season Column
season_map = {
    'Winter': 0, 'Summer': 1, 
    'Monsoon': 2, 'Post-Monsoon': 3
}

data['season'] = data['season'].map(season_map)


# Water source — ordinal risk order
ws_map = {
    'River': 0, 'Pond': 1, 'Open Well': 2, 'Rainwater': 3,
    'Tanker': 4, 'Borewell': 5, 'Piped': 6
}

data['water_source'] = data['water_source'].map(ws_map)


# Toilet access — binary
data['toilet_access'] = data['toilet_access'].map({0: 0, 1: 1})
if data['toilet_access'].dtype == object:
    data['toilet_access'] = LabelEncoder().fit_transform(data['toilet_access'])

print('Ordinal encoding done')
print(data[['handwashing_practice', 'water_treatment', 'water_source', 'season']].head())

Ordinal encoding done
   handwashing_practice  water_treatment  water_source  season
0                     0                0             5       3
1                     2                3             6       1
2                     1                0             4       3
3                     0                0             0       3
4                     2                1             0       3


### 3.3 Label Encoding (state)

In [31]:
le_state = LabelEncoder()
data['state'] = le_state.fit_transform(data['state'])

print('State encoding:')
for i, cls in enumerate(le_state.classes_):
    print(f'{cls} -> {i}')

State encoding:
Arunachal Pradesh -> 0
Assam -> 1
Manipur -> 2
Meghalaya -> 3
Mizoram -> 4
Nagaland -> 5
Sikkim -> 6
Tripura -> 7


### 3.4 Cyclical Encoding for Month

In [32]:
data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
data = data.drop(columns=['month'])

print('Month cyclical encoding done')
print(data[['month_sin', 'month_cos']].head())

Month cyclical encoding done
   month_sin     month_cos
0  -0.500000  8.660254e-01
1   1.000000  6.123234e-17
2  -0.866025  5.000000e-01
3  -0.866025  5.000000e-01
4  -0.500000  8.660254e-01


### 3.5 Log Transform Skewed Features